# Scalable LLM and Agent Systems: run every experiment

**Before you start:** Runtime -> Change runtime type -> **T4 GPU**. Then Runtime -> **Run all**.

1. The first cell clones the `v2-rebuild` branch from GitHub.
2. Everything else runs unattended (roughly 60-90 min on a T4).
3. The last cell downloads `results.zip`. Unzip it into the repo's `results/` folder.

Every number comes from a script in `scripts/` and is saved as JSON with library versions.

In [ ]:
import os, subprocess
print(subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout
      or "WARNING: no GPU. Runtime -> Change runtime type -> T4 GPU")
REPO, BRANCH = "https://github.com/PrathameshBende/Scalable-Multi-AI-Agents", "v2-rebuild"
if not os.path.exists("/content/project/.git"):
    subprocess.run(["git", "clone", "-q", "-b", BRANCH, REPO, "/content/project"], check=True)
os.chdir("/content/project")
!git pull -q && git log --oneline -1

In [ ]:
!pip install -q -r requirements.txt

## 1. Unit tests (no models needed)

In [ ]:
!python -m pytest -q

## 2. Start Ollama and pull the LLMs
qwen2.5:3b (~1.9 GB) for HyDE / RAG-Fusion query generation, qwen2.5:7b (~4.7 GB) for the agents.

In [ ]:
!apt-get -qq update > /dev/null && apt-get -qq install -y zstd pciutils lshw > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh
import subprocess, time, urllib.request
ollama = subprocess.Popen(["ollama", "serve"], stdout=open("/content/ollama.log", "w"), stderr=subprocess.STDOUT)
for _ in range(60):
    try:
        urllib.request.urlopen("http://127.0.0.1:11434"); print("Ollama is up"); break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError("Ollama did not start; see /content/ollama.log")

In [ ]:
!ollama pull qwen2.5:3b
!ollama pull qwen2.5:7b

## 3. Retrieval: dense baseline vs HyDE vs RAG-Fusion vs hybrid (SciFact, 300 queries)

In [ ]:
!python scripts/run_retrieval_eval.py --llm qwen2.5:3b

## 4. ReAct agent on 30 multi-step tasks (two model sizes)

In [ ]:
!python scripts/run_agent_eval.py --model qwen2.5:7b

In [ ]:
!python scripts/run_agent_eval.py --model qwen2.5:3b

## 5. Rebuilt RAG agent: end-to-end demo trace

In [ ]:
!python scripts/demo_rag_agent.py --model qwen2.5:7b --strategy hyde

## 6. ONNX: export + int8 quantization of the embedding model (CPU)

In [ ]:
ollama.terminate()  # free GPU memory for the next steps
!python scripts/run_onnx_eval.py

## 7. GGUF: Qwen2.5-0.5B -> F16 / Q8_0 / Q4_K_M with llama.cpp (build takes ~10 min)

In [ ]:
!python scripts/run_gguf_eval.py
!mkdir -p results/logs && cp work/gguf/*.log results/logs/

## 8. Summary and download

In [ ]:
!python scripts/summarize.py
import shutil
shutil.make_archive("/content/results", "zip", "/content/project/results")
from google.colab import files
files.download("/content/results.zip")